[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione-superiori/blob/main/01_il_mistero_del_valore.ipynb)

# Notebook 1 — Il mistero del valore di mercato

Siete il reparto dati di una società sportiva. Il direttore sportivo vi chiede di capire perché alcuni calciatori valgono pochi milioni e altri cifre enormi. In questo primo episodio non costruiamo ancora un modello predittivo: impariamo a guardare i dati come farebbe uno scout quantitativo.

I dati provengono da un dataset pubblico FIFA/EA Sports distribuito in formato CSV. Per rendere la lezione stabile, in questo corso non scarichiamo direttamente da Kaggle durante la lezione: usiamo una copia congelata in uno ZIP Dropbox del docente.

In [ ]:
from IPython.display import HTML, display

def concept(title, body, kind="idea"):
    colors = {
        "idea": ("#e8f3ff", "#0b5394"),
        "math": ("#fff3cd", "#7a4d00"),
        "warning": ("#fdecea", "#a61b1b"),
        "task": ("#eaf7ea", "#226622"),
        "story": ("#f3e8ff", "#5b2a86")
    }
    bg, border = colors.get(kind, colors["idea"])
    display(HTML(f'''
    <div style="background:{bg}; border-left:6px solid {border}; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
    <b style="color:{border}; font-size:18px">{title}</b><br>{body}
    </div>
    '''))

def reveal(title, body):
    display(HTML(f'''
    <details style="background:#f7f7f7; border:1px solid #ddd; padding:12px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
      <summary style="cursor:pointer; font-weight:bold">{title}</summary>
      <div style="margin-top:10px">{body}</div>
    </details>
    '''))

In [ ]:
concept("Missione", "Capire quali variabili sembrano collegate al valore di mercato di un calciatore. Non stiamo ancora facendo machine learning: stiamo costruendo intuizione sui dati.", "story")

In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

# Apriamo il file principale del dataset
csv_files = list(DATA_DIR.rglob("players_22.csv")) or list(DATA_DIR.rglob("*players*.csv"))
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Dataset caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

In [ ]:
concept("Dataset", "Ogni riga rappresenta un calciatore. Le colonne sono caratteristiche osservabili: eta', overall, potential, salario, club e valore economico. In un problema di regressione il valore economico sara' la variabile da prevedere, mentre le altre colonne saranno gli indizi.", "idea")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Come scriviamo il problema in matematica</b>

<p>Abbiamo $n$ giocatori. Per ognuno raccogliamo $p$ caratteristiche numeriche (et&agrave;, overall, potential, ...) e un valore di mercato.</p>

<ul>
<li>Le caratteristiche del giocatore $i$ formano un vettore $\mathbf{x}_i = (x_{i,1}, x_{i,2}, \dots, x_{i,p})$.</li>
<li>Il suo valore di mercato &egrave; un numero $y_i \in \mathbb{R}$.</li>
<li>Tutti i dati insieme: la <b>matrice degli input</b> $X \in \mathbb{R}^{n \times p}$ e il <b>vettore dei target</b> $\mathbf{y} \in \mathbb{R}^n$.</li>
</ul>

<p>Un problema di <b>regressione</b> consiste nel trovare una funzione $f$ tale che $f(\mathbf{x}_i) \approx y_i$ per tutti i giocatori. Il termine $\approx$ &egrave; importante: non vogliamo (e non possiamo) prevedere il valore esatto, vogliamo prevederlo <i>bene in media</i>.</p>
</div>


## Primo sguardo: chi sono i giocatori più costosi?

In [ ]:
df.sort_values("value_eur", ascending=False)[["short_name", "age", "overall", "potential", "value_mln_eur", "club_name", "player_positions"]].head(15)

In [ ]:
reveal("Domanda per la classe", "Guardando questa tabella, quale variabile vi sembra piu' importante: eta', overall o potential? Scrivete un'ipotesi prima di guardare i grafici.")

## Grafico 1: età e valore

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["age"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Eta'")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Eta' vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
concept("Relazione tra variabili", "Quando disegniamo un grafico con una variabile sull'asse x e una sull'asse y, stiamo chiedendo se conoscere x ci aiuta a prevedere y. Se i punti formano una struttura riconoscibile, allora x contiene informazione utile su y. Se i punti sono completamente sparsi, x da sola probabilmente non basta.", "idea")

## Grafico 2: overall e valore

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["overall"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Overall rating")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Overall vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

## Grafico 3: potential e valore

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["potential"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Potential rating")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Potential vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
concept("Correlazione intuitiva", "Se al crescere di una variabile tende a crescere anche il valore, diciamo che c'e' una correlazione positiva. Attenzione: correlazione non significa causalita'. Qui non stiamo dicendo che aumentare artificialmente l'overall causa automaticamente un prezzo piu' alto; stiamo osservando che nel dataset i due numeri si muovono spesso insieme.", "warning")

## Una misura numerica dell'associazione

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Coefficiente di correlazione</b>

<p>Per misurare quanto due variabili numeriche si muovono insieme usiamo il <b>coefficiente di correlazione di Pearson</b>:</p>

$$r_{x,y} \;=\; \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}\;\sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

<p>dove $\bar{x}$ e $\bar{y}$ sono le medie. Il valore $r$ sta sempre nell'intervallo $[-1, +1]$:</p>

<ul>
<li>$r \approx +1$: al crescere di $x$ tende a crescere anche $y$ (correlazione positiva).</li>
<li>$r \approx -1$: al crescere di $x$, $y$ tende a calare (correlazione negativa).</li>
<li>$r \approx 0$: nessuna relazione <i>lineare</i> evidente (potrebbe esserci una relazione non lineare).</li>
</ul>

<p><b>Attenzione</b>: una correlazione alta non implica una relazione di causa-effetto.</p>
</div>


In [ ]:
numeric_cols = [c for c in ["age", "overall", "potential", "wage_eur", "value_mln_eur"] if c in df.columns]
corr = df[numeric_cols].corr(numeric_only=True)["value_mln_eur"].sort_values(ascending=False)
corr

In [ ]:
reveal("Cosa dovremmo aver capito", "Il valore di mercato non dipende da una sola variabile. Overall e potential sono molto informativi, l'eta' ha un effetto piu' sottile, e il salario e' spesso collegato al valore. Nel prossimo notebook costruiremo il primo modello: una formula che prova a prevedere il valore.")